# Phase 1 Final Reporting Package

This notebook is orchestration only. It runs the repository CLI for the final reporting package after final governance reconciliation has been reviewed.

Scientific integrity rules:

- This notebook does not train or retrain any comparator.
- This notebook does not recompute logits, controls, calibration, influence, or efficacy metrics.
- The reporting runner records existing governance status and blockers exactly as read from the selected governance reconciliation run.
- A complete reporting package may be claim-evaluable as reporting, but it does not make failed controls, calibration, or influence pass.
- Headline Phase 1 claims remain closed unless a later final governance reconciliation shows every required surface claim-evaluable under preregistered rules.


In [ ]:
# Cell 1 - Bootstrap repo and run unit tests.

from pathlib import Path
import base64
import getpass
import json
import os
import subprocess
import sys

try:
    from google.colab import drive  # type: ignore
    drive.mount('/content/drive')
except Exception:
    pass

REPO_URL = 'https://github.com/BrianNguyen29/eeg-ds004752.git'
REPO_DIR = Path('/content/eeg-ds004752')

def run(cmd, cwd=None, check=True):
    display = []
    for item in cmd:
        text = str(item)
        if 'Authorization: Basic' in text:
            display.append('http.extraHeader=<redacted>')
        else:
            display.append(text)
    print('$ ' + ' '.join(display))
    result = subprocess.run(cmd, cwd=str(cwd) if cwd else None, text=True, check=check)
    return result

if not REPO_DIR.exists():
    token = getpass.getpass('Nhap GitHub token cho repo private: ')
    header = 'Authorization: Basic ' + base64.b64encode(f'x-access-token:{token}'.encode()).decode()
    run(['git', '-c', f'http.extraHeader={header}', 'clone', REPO_URL, str(REPO_DIR)])
else:
    run(['git', 'pull', '--ff-only'], cwd=REPO_DIR)

run(['git', 'log', '--oneline', '-5'], cwd=REPO_DIR)
unit_result = subprocess.run(['python', '-m', 'unittest', 'discover', '-s', 'tests'], cwd=str(REPO_DIR), text=True)
if unit_result.returncode != 0:
    raise subprocess.CalledProcessError(unit_result.returncode, ['python', '-m', 'unittest', 'discover', '-s', 'tests'])


In [ ]:
# Cell 2 - Select reviewed source artifacts and keep launch disabled by default.

from pathlib import Path
import hashlib
import json

DRIVE_ROOT = Path('/content/drive/MyDrive/eeg-ds004752')
ARTIFACT_ROOT = DRIVE_ROOT / 'artifacts'

EXPECTED_PREREG_IDENTITY_HASH = '87e928ea747099c336a32121bc156655a1a160d666a251c7ac41228efba96af6'
PREREG_BUNDLE = ARTIFACT_ROOT / 'prereg/20260418T161442014597Z/prereg_bundle.json'

# Use None to resolve latest.txt, or pin an already reviewed final governance reconciliation run.
GOVERNANCE_RECONCILIATION_RUN = Path('/content/drive/MyDrive/eeg-ds004752/artifacts/phase1_final_governance_reconciliation/20260422T071329009670Z')

OUTPUT_ROOT = ARTIFACT_ROOT / 'phase1_final_reporting'
PLAN_ROOT = ARTIFACT_ROOT / 'phase1_final_reporting_plan'

RUN_FINAL_REPORTING = False
REQUIRED_ACK = 'I reviewed final reporting and understand it records closed claims without fabricating missing governance evidence'
MANUAL_LAUNCH_ACK = ''

FIX_MARKER = 'phase1_final_reporting_v1_20260422'
print('Notebook fix marker:', FIX_MARKER)


In [ ]:
# Cell 3 - Resolve paths and validate prereg identity plus final governance boundary.

def read_json(path: Path):
    return json.loads(path.read_text(encoding='utf-8'))

def sha256(path: Path) -> str:
    h = hashlib.sha256()
    with path.open('rb') as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b''):
            h.update(chunk)
    return h.hexdigest()

def resolve_latest(root: Path) -> Path:
    latest = root / 'latest.txt'
    if latest.exists():
        return Path(latest.read_text(encoding='utf-8').strip())
    runs = sorted([p for p in root.iterdir() if p.is_dir()]) if root.exists() else []
    if not runs:
        raise FileNotFoundError(f'No runs found under {root}')
    return runs[-1]

def resolve_prereg_bundle(path: Path) -> Path:
    path = Path(path)
    if path.exists():
        return path
    candidates = sorted(ARTIFACT_ROOT.glob('prereg/*/prereg_bundle.json'))
    print('Configured prereg bundle not found:', path)
    print('Found prereg bundles:', len(candidates))
    for item in candidates[-5:]:
        print('candidate:', item)
    if not candidates:
        raise FileNotFoundError(f'No prereg_bundle.json found under {ARTIFACT_ROOT / "prereg"}')
    return candidates[-1]

PREREG_BUNDLE = resolve_prereg_bundle(Path(PREREG_BUNDLE))
GOVERNANCE_RECONCILIATION_RUN = Path(GOVERNANCE_RECONCILIATION_RUN) if GOVERNANCE_RECONCILIATION_RUN else resolve_latest(ARTIFACT_ROOT / 'phase1_final_governance_reconciliation')

bundle = read_json(PREREG_BUNDLE)
actual_prereg_hash = bundle.get('prereg_bundle_hash_sha256')
raw_prereg_sha256 = sha256(PREREG_BUNDLE)
print('Prereg status:', bundle.get('status'))
print('Locked prereg identity hash:', actual_prereg_hash)
print('Raw prereg file SHA256:', raw_prereg_sha256)
assert bundle.get('status') == 'locked', 'Prereg bundle must be locked.'
assert actual_prereg_hash == EXPECTED_PREREG_IDENTITY_HASH, 'Locked prereg identity hash mismatch.'

gov_summary = read_json(GOVERNANCE_RECONCILIATION_RUN / 'phase1_final_governance_reconciliation_summary.json')
gov_claim_state = read_json(GOVERNANCE_RECONCILIATION_RUN / 'phase1_final_governance_claim_state.json')
assert gov_summary.get('comparator_outputs_complete') is True
assert gov_summary.get('runtime_logs_audited_for_all_required_comparators') is True
assert gov_summary.get('claim_ready') is False
assert gov_summary.get('headline_phase1_claim_open') is False
assert gov_summary.get('full_phase1_claim_bearing_run_allowed') is False
assert gov_claim_state.get('claim_ready') is False
print('Governance reconciliation run:', GOVERNANCE_RECONCILIATION_RUN)
print('Governance status:', gov_summary.get('status'))
print('Current governance blockers:', gov_summary.get('claim_blockers'))


In [ ]:
# Cell 4 - Preflight runner/config availability and write a reviewable plan artifact.

from datetime import datetime, timezone

sys.path.insert(0, str(REPO_DIR))
preflight_blockers = []
try:
    from src.phase1.final_reporting import run_phase1_final_reporting  # noqa: F401
    runner_available = True
except Exception as exc:
    runner_available = False
    preflight_blockers.append(f'final reporting runner import failed: {exc}')

required_repo_files = [
    REPO_DIR / 'src/phase1/final_reporting.py',
    REPO_DIR / 'configs/phase1/final_reporting.json',
    REPO_DIR / 'configs/phase1/final_governance_reconciliation.json',
]
for path in required_repo_files:
    if not path.exists():
        preflight_blockers.append(f'missing repo file: {path.relative_to(REPO_DIR)}')

PLAN_ROOT.mkdir(parents=True, exist_ok=True)
plan_dir = PLAN_ROOT / datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
plan_dir.mkdir(parents=True, exist_ok=False)

cmd = [
    'python', '-m', 'src.cli', 'phase1_final_reporting',
    '--profile', 't4_safe',
    '--config', str(PREREG_BUNDLE),
    '--governance-reconciliation-run', str(GOVERNANCE_RECONCILIATION_RUN),
    '--output-root', str(OUTPUT_ROOT),
    '--reporting-config', 'configs/phase1/final_reporting.json',
    '--governance-config', 'configs/phase1/final_governance_reconciliation.json',
]

plan = {
    'status': 'phase1_final_reporting_plan_recorded',
    'created_utc': plan_dir.name,
    'fix_marker': FIX_MARKER,
    'runner_available': runner_available,
    'run_flag': RUN_FINAL_REPORTING,
    'required_ack': REQUIRED_ACK,
    'ack_matched': MANUAL_LAUNCH_ACK == REQUIRED_ACK,
    'preflight_blockers': preflight_blockers,
    'prereg_bundle': str(PREREG_BUNDLE),
    'locked_prereg_identity_hash': actual_prereg_hash,
    'raw_prereg_file_sha256': raw_prereg_sha256,
    'governance_reconciliation_run': str(GOVERNANCE_RECONCILIATION_RUN),
    'output_root': str(OUTPUT_ROOT),
    'command': cmd,
    'scientific_integrity_rule': 'Plan only. The reporting package must preserve upstream blockers and keep all claims closed.',
}
(plan_dir / 'phase1_final_reporting_plan.json').write_text(json.dumps(plan, indent=2) + '
', encoding='utf-8')
print(json.dumps({k: plan[k] for k in ['status', 'runner_available', 'run_flag', 'ack_matched', 'preflight_blockers']}, indent=2))
if preflight_blockers:
    raise RuntimeError(preflight_blockers)


In [ ]:
# Cell 5 - Manual hold or launch the CLI-backed final reporting runner.

if not RUN_FINAL_REPORTING:
    hold = {
        'status': 'phase1_final_reporting_manual_hold',
        'run_flag': RUN_FINAL_REPORTING,
        'required_ack': REQUIRED_ACK,
        'ack_matched': MANUAL_LAUNCH_ACK == REQUIRED_ACK,
        'plan_dir': str(plan_dir),
    }
    print(json.dumps(hold, indent=2))
elif MANUAL_LAUNCH_ACK != REQUIRED_ACK:
    raise RuntimeError('Manual launch acknowledgement mismatch. Do not launch final reporting without explicit claim-closed acknowledgement.')
else:
    launch_review = {
        'status': 'phase1_final_reporting_launch_review_passed',
        'run_flag': RUN_FINAL_REPORTING,
        'ack_matched': True,
        'claim_ready_before_run': False,
        'upstream_governance_status': gov_summary.get('status'),
    }
    (plan_dir / 'phase1_final_reporting_launch_review.json').write_text(json.dumps(launch_review, indent=2) + '
', encoding='utf-8')
    run(cmd, cwd=REPO_DIR)
    print('Final reporting command completed. Review generated artifacts before downstream governance reconciliation.')


In [ ]:
# Cell 6 - Inspect latest final reporting output if execution was launched.

summary = None
latest_run = None
if RUN_FINAL_REPORTING:
    latest_run = resolve_latest(OUTPUT_ROOT)
    print('Latest final reporting output:', latest_run)
    required_files = [
        'phase1_final_reporting_inputs.json',
        'phase1_final_reporting_summary.json',
        'phase1_final_reporting_report.md',
        'phase1_final_reporting_source_links.json',
        'phase1_final_reporting_input_validation.json',
        'final_comparator_completeness_table.json',
        'negative_controls_report.json',
        'calibration_package_report.json',
        'influence_package_report.json',
        'final_fold_logs.json',
        'claim_state_report.json',
        'main_phase1_report.md',
        'phase1_final_reporting_claim_table.json',
        'final_reporting_manifest.json',
        'phase1_final_reporting_claim_state.json',
    ]
    for name in required_files:
        print(name, 'exists =', (latest_run / name).exists())
    summary = read_json(latest_run / 'phase1_final_reporting_summary.json')
    print(json.dumps({
        'status': summary.get('status'),
        'reporting_package_passed': summary.get('reporting_package_passed'),
        'claim_table_ready': summary.get('claim_table_ready'),
        'claims_opened': summary.get('claims_opened'),
        'upstream_governance_status': summary.get('upstream_governance_status'),
        'upstream_governance_blocked': summary.get('upstream_governance_blocked'),
        'claim_blockers': summary.get('claim_blockers'),
    }, indent=2))
else:
    print('Manual hold only; no final reporting output inspected.')


In [ ]:
# Cell 7 - Assertions and non-claim review note.

if RUN_FINAL_REPORTING:
    assert summary is not None
    assert summary.get('claim_ready') is False
    assert summary.get('headline_phase1_claim_open') is False
    assert summary.get('full_phase1_claim_bearing_run_allowed') is False
    assert summary.get('claims_opened') is False
    assert summary.get('status') in ['phase1_final_reporting_complete_claim_closed', 'phase1_final_reporting_blocked_claim_closed']
    manifest = read_json(latest_run / 'final_reporting_manifest.json')
    claim_table = read_json(latest_run / 'phase1_final_reporting_claim_table.json')
    assert manifest.get('claim_ready') is False
    assert manifest.get('claims_opened') is False
    assert manifest.get('smoke_artifacts_promoted') is False
    assert claim_table.get('claim_table_ready') is True
    assert claim_table.get('claims_opened') is False
    assert all(row.get('claim_open') is False for row in claim_table.get('rows', []))
    review_note = {
        'status': 'phase1_final_reporting_review_recorded',
        'reviewed_run': str(latest_run),
        'scope': 'final reporting package from final governance reconciliation artifacts',
        'reporting_package_passed': summary.get('reporting_package_passed'),
        'upstream_governance_blocked': summary.get('upstream_governance_blocked'),
        'metrics_interpretation': {
            'allowed': 'Reporting/provenance and closed-claim table only.',
            'not_allowed': 'Do not treat this report as decoder efficacy, privileged-transfer efficacy, or full Phase 1 evidence.',
        },
        'next_allowed_steps': [
            'Rerun final governance reconciliation with final_reporting_manifest if review is acceptable.',
            'Do not open headline claims while controls/calibration/influence blockers remain.',
        ],
        'not_ok_to_claim': [
            'decoder efficacy',
            'A2d efficacy',
            'A3/A4 efficacy',
            'A4 superiority',
            'privileged-transfer efficacy',
            'full Phase 1 neural comparator performance',
        ],
    }
    (latest_run / 'phase1_final_reporting_review_note.json').write_text(json.dumps(review_note, indent=2) + '
', encoding='utf-8')
    print('Review note written:', latest_run / 'phase1_final_reporting_review_note.json')
    print(json.dumps(review_note, indent=2))
else:
    print('No assertions beyond plan/manual hold because RUN_FINAL_REPORTING is False.')


In [ ]:
# Cell 8 - Closeout.

print('================ Phase 1 Final Reporting Closeout ================')
print('Notebook fix marker:', FIX_MARKER)
print('Plan source of truth:', plan_dir)
print('Runner available:', runner_available)
print('Run flag:', RUN_FINAL_REPORTING)
print('Expected locked prereg identity hash:', EXPECTED_PREREG_IDENTITY_HASH)
print('Locked prereg hash from bundle:', actual_prereg_hash)
print('Raw prereg file SHA256:', raw_prereg_sha256)
print('Governance reconciliation run:', GOVERNANCE_RECONCILIATION_RUN)
print('Preflight blockers:', preflight_blockers)
if RUN_FINAL_REPORTING:
    print('Latest final reporting output:', latest_run)
    print('Reporting package passed:', summary.get('reporting_package_passed'))
    print('Claim table ready:', summary.get('claim_table_ready'))
    print('Claims opened:', summary.get('claims_opened'))
    print('Claim blockers:', summary.get('claim_blockers'))
    print('CHECK REQUIRED: Review final_reporting_manifest.json and claim table before rerunning final governance reconciliation with --final-reporting-manifest.')
else:
    print('HELD: Runner appears available, but reporting was not launched because manual flag is False.')
    print('NEXT: review the plan, then rerun only with explicit claim-closed acknowledgement if appropriate.')
print('NOT OK TO CLAIM: This notebook does not prove decoder efficacy, A2d efficacy, A3/A4 efficacy, A4 superiority, privileged-transfer efficacy, or full Phase 1 neural comparator performance.')
